### SetUp

In [ ]:
# !mkdir -p /content/Data

# !gdown --id 1ZGTOzUuXfpeoe2r6rBVxY7hGzf9JjUf6 -O Data/B3_C1_D2_4395_4417.mp4
# !gdown --id 11wS44lgFVhJZm7fnU9N0oLOutVe5yT-k -O Data/B5_C1_D2_4526_4485.mp4
# !gdown --id 1wdz-SyOmgra4O53Ac_yFTUCp1gBsg0Pj -O Data/B5_C2_D1_4492_4499.mp4

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1ZGTOzUuXfpeoe2r6rBVxY7hGzf9JjUf6
From (redirected): https://drive.google.com/uc?id=1ZGTOzUuXfpeoe2r6rBVxY7hGzf9JjUf6&confirm=t&uuid=3bee3c7c-939b-49eb-9861-41769f427b25
To: /content/Data/B3_C1_D2_4395_4417.mp4
100% 411M/411M [00:06<00:00, 67.0MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=11wS44lgFVhJZm7fnU9N0oLOutVe5yT-k
From (redirected): https://drive.google.com/uc?id=11wS44lgFVhJZm7fnU9N0oLOutVe5yT-k&confirm=t&uuid=c6c7c9f4-ebda-47c4-809e-c03a67f99e20
To: /content

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

# !cp -r /content/drive/MyDrive/Data /content/

Mounted at /content/drive


In [ ]:
!mkdir -p /content/Data

!gdown --id 12vyxgh8-dL9qEWkNs4-HUy-5FtDm7gTu -O Data/B3_C1_D2_4395_4417_trimmed_resized.mp4
!gdown --id 1H6x-mTD824SrYXnwqCalXvAzj6P65HoI -O Data/B5_C1_D2_4526_4485_trimmed_resized.mp4
!gdown --id 1lrktZmDAsmVPPRH3dqQnE9qtaFYs_ief -O Data/B5_C2_D1_4492_4499_trimmed_resized.mp4

In [2]:
!gdown https://drive.google.com/uc?id=1YTbpI-m27oDJdGivKA1x01MA37Kv8_5- -O /content/pigs-yolov8-weights.zip
!unzip /content/pigs-yolov8-weights.zip -d /content

Downloading...
From (original): https://drive.google.com/uc?id=1YTbpI-m27oDJdGivKA1x01MA37Kv8_5-
From (redirected): https://drive.google.com/uc?id=1YTbpI-m27oDJdGivKA1x01MA37Kv8_5-&confirm=t&uuid=266915d4-9e4e-4c93-bf5e-5887bebbc47d
To: /content/pigs-yolov8-weights.zip
100% 169M/169M [00:05<00:00, 31.4MB/s]
Archive:  /content/pigs-yolov8-weights.zip
  inflating: /content/v9/labels.jpg  
  inflating: /content/v9/train_batch5040.jpg  
  inflating: /content/v9/results.png  
  inflating: /content/v9/confusion_matrix_normalized.png  
  inflating: /content/v9/confusion_matrix.png  
  inflating: /content/v9/train_batch1.jpg  
  inflating: /content/v9/PR_curve.png  
  inflating: /content/v9/P_curve.png  
  inflating: /content/v9/events.out.tfevents.1718383799.Manzar-PC.2885.0  
  inflating: /content/v9/val_batch1_labels.jpg  
  inflating: /content/v9/train_batch0.jpg  
  inflating: /content/v9/train_batch5042.jpg  
  inflating: /content/v9/train_batch5041.jpg  
  inflating: /content/v9/train_b

In [3]:
!pip install -qU ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.0 MB/s eta 0:00:00


### YOLO + SAM2 Video Segmentation Pipeline

In this notebook, we will build a pipeline that:

1. **Trims** a video to the first $N$ frames (default: $250$).
2. **Detects objects** in the first frame using YOLOv8.
3. **Segments** the detected objects across the trimmed video using SAM2.
4. **Saves** the final segmented video as output.

This approach ensures faster processing and prevents out-of-memory errors when working with long videos.


In [8]:
import os
import gc
import cv2
import torch
import torchvision.transforms.functional as F
from ultralytics import YOLO
from ultralytics.models.sam import SAM2VideoPredictor
from tqdm import tqdm

# -----------------------------
# Config
# -----------------------------
YOLO_MODEL_PATH = "/content/v9/weights/best.pt"
SAM_MODEL_PATH = "sam2.1_b.pt"
INPUT_DIR = "/content/Data"
VIDEO_LIST = [
    "B5_C2_D1_4492_4499_trimmed_resized.mp4",
    "B5_C1_D2_4526_4485_trimmed_resized.mp4",
    "B3_C1_D2_4395_4417_trimmed_resized.mp4"
]

# -----------------------------
# Load models
# -----------------------------
yolo_model = YOLO(YOLO_MODEL_PATH)
overrides = dict(conf=0.25, task='segment', mode='predict', imgsz=1024, model=SAM_MODEL_PATH)
predictor = SAM2VideoPredictor(overrides=overrides)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
# -----------------------------
# Helper: Trim and resize video
# -----------------------------

def trim_and_resize_video(input_path, skip_start=10, keep_frames=-1, scale=0.5, skip_step=1):
    """
    Trim video, resize, and allow skipping frames.

    Args:
        input_path (str): Path to input video
        skip_start (int): Number of initial frames to skip
        keep_frames (int): Number of frames to keep after skipping (-1 = keep all remaining frames)
        scale (float): Fraction to resize width and height (0.5 = half size)
        skip_step (int): Skip every N frames after saving one (0 = keep all, 1 = save 1, skip 1, etc.)

    Returns:
        str: Path to saved trimmed & resized video
    """
    # Output path
    base, ext = os.path.splitext(input_path)
    output_path = f"{base}_trimmed_resized{ext}"

    # Open video
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print("❌ Cannot open video")
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) * scale)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) * scale)

    # Video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    saved_count = 0

    # Total frames to process for progress bar
    if keep_frames == -1:
        expected_frames = max(0, total_frames - skip_start)
    else:
        expected_frames = min(max(0, total_frames - skip_start), keep_frames)

    with tqdm(total=expected_frames, desc="Processing video", unit="frame") as pbar:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if frame_count >= skip_start:
                # Only process frames according to skip_step
                if (frame_count - skip_start) % (skip_step + 1) == 0:
                    frame_resized = cv2.resize(frame, (width, height))
                    out.write(frame_resized)
                    saved_count += 1
                    pbar.update(1)

                    # Stop if reached keep_frames (unless -1 = all frames)
                    if keep_frames != -1 and saved_count >= keep_frames:
                        break

            frame_count += 1

    cap.release()
    out.release()
    print(f"✅ Trimmed + resized video saved: {output_path} ({saved_count} frames kept)")
    return output_path


In [ ]:
# # -----------------------------
# # Run pipeline on video list
# # -----------------------------
# for video_name in VIDEO_LIST:
#     video_path = f"{INPUT_DIR}/{video_name}"

#     # Step 1: Preprocess (trim + resize once)
#     processed_path = trim_and_resize_video(
#         input_path=video_path,
#         skip_start=10,
#         keep_frames=-1,
#         scale=0.5,
#         skip_step=1
#     )

Processing video:  50%|█████     | 5394/10787 [03:56<03:56, 22.78frame/s]


✅ Trimmed + resized video saved: /content/Data/B5_C2_D1_4492_4499_trimmed_resized.mp4 (5394 frames kept)


Processing video:  50%|█████     | 5390/10780 [03:54<03:54, 22.98frame/s]


✅ Trimmed + resized video saved: /content/Data/B5_C1_D2_4526_4485_trimmed_resized.mp4 (5390 frames kept)


Processing video:  50%|█████     | 4736/9471 [03:30<03:30, 22.47frame/s]

✅ Trimmed + resized video saved: /content/Data/B3_C1_D2_4395_4417_trimmed_resized.mp4 (4736 frames kept)


In [4]:
!ls /content/Data -alh

total 2.3G
drwx------ 4 root root 4.0K Aug 28 23:24 .
drwxr-xr-x 1 root root 4.0K Aug 28 23:24 ..
-rw------- 1 root root 393M Aug 28 23:23 B3_C1_D2_4395_4417.mp4
-rw------- 1 root root 283M Aug 28 23:24 B3_C1_D2_4395_4417_trimmed_resized.mp4
-rw------- 1 root root 447M Aug 28 23:23 B5_C1_D2_4526_4485.mp4
-rw------- 1 root root 351M Aug 28 23:24 B5_C1_D2_4526_4485_trimmed_resized.mp4
-rw------- 1 root root 447M Aug 28 23:24 B5_C2_D1_4492_4499.mp4
-rw------- 1 root root 346M Aug 28 23:24 B5_C2_D1_4492_4499_trimmed_resized.mp4
drwx------ 2 root root 4.0K Aug 28 23:24 .ipynb_checkpoints
drwx------ 4 root root 4.0K Aug 28 23:24 segment


In [6]:
import gc
import tempfile
import shutil

def process_video_in_chunks(processed_path, chunk_size=750):
    """
    Process a video in true chunks:
      1. Break video into chunks of `chunk_size` frames
      2. Run YOLO on the first frame of each chunk
      3. Run SAM segmentation for that chunk
      4. Clear GPU and delete temporary chunk
    """

    # ---- Create temporary folder ----
    temp_dir = tempfile.mkdtemp()
    print(f"Temporary folder for chunks: {temp_dir}")

    cap = cv2.VideoCapture(processed_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"Processing video: {processed_path}, total frames: {total_frames}")

    chunk_count = 0
    frames_buffer = []

    while True:
        ret, frame = cap.read()
        if not ret:
            if frames_buffer:
                # ---- Process last chunk ----
                chunk_count += 1
                temp_path = os.path.join(temp_dir, f"chunk_{chunk_count}.mp4")
                fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                out = cv2.VideoWriter(temp_path, fourcc, fps, (width, height))
                for f in frames_buffer:
                    out.write(f)
                out.release()

                # YOLO on first frame
                results = yolo_model(frames_buffer[0], verbose=False)
                points, labels = [], []
                for box in results[0].boxes:
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    cx, cy = (x1 + x2)/2, (y1 + y2)/2
                    points.append([cx, cy])
                    labels.append(1)

                if points:
                    predictor(
                        source=temp_path,
                        points=points,
                        labels=labels
                    )

                torch.cuda.empty_cache()
                gc.collect()
                del frames_buffer
                os.remove(temp_path)
            break

        frames_buffer.append(frame)

        if len(frames_buffer) >= chunk_size:
            # ---- Save current chunk ----
            chunk_count += 1
            temp_path = os.path.join(temp_dir, f"chunk_{chunk_count}.mp4")
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(temp_path, fourcc, fps, (width, height))
            for f in frames_buffer:
                out.write(f)
            out.release()

            # YOLO on first frame
            results = yolo_model(frames_buffer[0], verbose=False)
            points, labels = [], []
            for box in results[0].boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                cx, cy = (x1 + x2)/2, (y1 + y2)/2
                points.append([cx, cy])
                labels.append(1)

            if points:
                predictor(
                    source=temp_path,
                    points=points,
                    labels=labels
                )

            print(f"✅ Processed chunk {chunk_count} ({len(frames_buffer)} frames)")

            # ---- Clear GPU and memory ----
            torch.cuda.empty_cache()
            gc.collect()
            del frames_buffer
            os.remove(temp_path)

            frames_buffer = []

    cap.release()
    shutil.rmtree(temp_dir)
    print("✅ Video processing complete")

In [9]:
for video_name in VIDEO_LIST:
    base_name, ext = os.path.splitext(video_name)
    video_path = os.path.join(INPUT_DIR, f"{base_name}_trimmed_resized{ext}")

    # Step 2: Process in chunks with YOLO + SAM
    process_video_in_chunks(
        processed_path=video_path,
        chunk_size=750
    )

    break

Streaming output truncated to the last 5000 lines.
video 1/1 (frame 503/750) /tmp/tmp4wzn12wr/chunk_1.mp4: 1024x1024 1 0, 1 1, 864.9ms
video 1/1 (frame 504/750) /tmp/tmp4wzn12wr/chunk_1.mp4: 1024x1024 1 0, 1 1, 861.7ms
video 1/1 (frame 505/750) /tmp/tmp4wzn12wr/chunk_1.mp4: 1024x1024 1 0, 1 1, 861.4ms
video 1/1 (frame 506/750) /tmp/tmp4wzn12wr/chunk_1.mp4: 1024x1024 1 0, 1 1, 859.5ms
video 1/1 (frame 507/750) /tmp/tmp4wzn12wr/chunk_1.mp4: 1024x1024 1 0, 1 1, 869.0ms
video 1/1 (frame 508/750) /tmp/tmp4wzn12wr/chunk_1.mp4: 1024x1024 1 0, 1 1, 860.3ms
video 1/1 (frame 509/750) /tmp/tmp4wzn12wr/chunk_1.mp4: 1024x1024 1 0, 1 1, 865.0ms
video 1/1 (frame 510/750) /tmp/tmp4wzn12wr/chunk_1.mp4: 1024x1024 1 0, 1 1, 863.4ms
video 1/1 (frame 511/750) /tmp/tmp4wzn12wr/chunk_1.mp4: 1024x1024 1 0, 1 1, 867.5ms
video 1/1 (frame 512/750) /tmp/tmp4wzn12wr/chunk_1.mp4: 1024x1024 1 0, 1 1, 863.1ms
video 1/1 (frame 513/750) /tmp/tmp4wzn12wr/chunk_1.mp4: 1024x1024 1 0, 1 1, 865.5ms
video 1/1 (frame 514/750)

In [18]:
!ls -al /content/runs/segment/predict

total 2318384
drwxr-xr-x 2 root root      4096 Aug 29 01:00 .
drwxr-xr-x 3 root root      4096 Aug 28 23:27 ..
-rw-r--r-- 1 root root 269188450 Aug 28 23:38 chunk_1.avi
-rw-r--r-- 1 root root 268653388 Aug 28 23:50 chunk_2.avi
-rw-r--r-- 1 root root 269685400 Aug 29 00:02 chunk_3.avi
-rw-r--r-- 1 root root 271658610 Aug 29 00:13 chunk_4.avi
-rw-r--r-- 1 root root 273227818 Aug 29 00:25 chunk_5.avi
-rw-r--r-- 1 root root 272906776 Aug 29 00:36 chunk_6.avi
-rw-r--r-- 1 root root 272697400 Aug 29 00:48 chunk_7.avi
-rw-r--r-- 1 root root  52383954 Aug 29 00:50 chunk_8.avi
-rw-r--r-- 1 root root 423575662 Aug 29 00:57 final_output.avi


### Merge Chunk

In [16]:
def merge_chunks(chunks_dir, output_file="final_output.avi"):
    """
    Merge all chunk_*.avi files in a directory into one final video with a progress bar.

    Args:
        chunks_dir (str): Path to the directory containing chunk_*.avi files
        output_file (str): Name of the merged video file (default: final_output.avi)
    """

    # Collect all chunk files and sort them in order
    chunk_files = sorted(
        [f for f in os.listdir(chunks_dir) if f.startswith("chunk_") and f.endswith(".avi")],
        key=lambda x: int(x.split("_")[1].split(".")[0])
    )

    if not chunk_files:
        print("❌ No chunk files found in directory.")
        return

    print("Merging chunks in order:", chunk_files)

    # Read first chunk to get FPS, width, height
    first_cap = cv2.VideoCapture(os.path.join(chunks_dir, chunk_files[0]))
    fps = int(first_cap.get(cv2.CAP_PROP_FPS))
    width = int(first_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(first_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    first_cap.release()

    # Always save inside chunks_dir
    output_path = os.path.join(chunks_dir, output_file)

    # Define video writer
    fourcc = cv2.VideoWriter_fourcc(*'XVID')  # use 'mp4v' if you prefer .mp4
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Count total frames for progress bar
    total_frames = 0
    for file in chunk_files:
        cap = cv2.VideoCapture(os.path.join(chunks_dir, file))
        total_frames += int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()

    # Write frames with progress bar
    with tqdm(total=total_frames, desc="Merging video", unit="frame") as pbar:
        for file in chunk_files:
            cap = cv2.VideoCapture(os.path.join(chunks_dir, file))
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                out.write(frame)
                pbar.update(1)
            cap.release()

    out.release()
    print(f"🎉 Final video saved at: {output_path}")

In [15]:
merge_chunks("/content/runs/segment/predict", "final_output.avi")

Merging chunks in order: ['chunk_1.avi', 'chunk_2.avi', 'chunk_3.avi', 'chunk_4.avi', 'chunk_5.avi', 'chunk_6.avi', 'chunk_7.avi', 'chunk_8.avi']


🔄 Merging video: 100%|██████████| 5394/5394 [01:47<00:00, 50.02frame/s]

🎉 Final video saved at: final_output.avi


In [17]:
!mv /content/final_output.avi /content/runs/segment/predict

### Save In Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# !mkdir -p /content/drive/MyDrive/Data

In [ ]:
# !cp -r /content/Data /content/drive/MyDrive/

In [ ]:
# !ls -al /content/runs/segment/predict2

In [19]:
!cp -r /content/runs/segment/ /content/drive/MyDrive/Data